# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdsvr/flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

**Design: features from March 2026, label observed in April 2026** — a real past->future setup, so leakage has somewhere real to hide (that's the point of this notebook).

- **Population:** the Lane 4 slice — content items with >=100 March impressions at a real position (1-20), GSC-measured rows only. Plus one **disclosed outcome-window selection**: items must also have >=50 April impressions so an April CTR is measurable. That filter peeks at the outcome window — it's a population choice, not a feature, and it's disclosed here per the leakage skill (limitations, not crimes).
- **Label (observed, not defined):** `ctr_apr < 0.8 * ctr_mar` — the item's CTR dropped >=20% in the next 30 days. Measured later in time; no rule of mine defines it into existence.
- **Engineered features (all March-or-earlier):** log impressions, March CTR, impression-weighted position, days-with-impressions, content age, days since last update, `has_word_count` + filled word count, one-hot content type.
- **Missing handling:** word_count missingness is *patterned* (proved in section 2), so a `has_word_count` flag carries the pattern honestly instead of a blind `fillna(0)` inventing it as content length.

In [2]:
# Import ML libs FIRST — on a low-RAM machine, importing sklearn after DuckDB has
# grabbed its buffer pool can fail with a paging-file MemoryError.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, GroupShuffleSplit
import os
import duckdb
import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

con = duckdb.connect()
con.execute("SET memory_limit='1500MB'")
con.execute("SET temp_directory='data/duckdb_spill'")  # spill instead of OOM
con.execute("SET preserve_insertion_order=false")
MAR = "'data/warehouse/fact/month=2026-03/data_0.parquet'"
APR = "'data/warehouse/fact/month=2026-04/data_0.parquet'"
DIM = "'data/warehouse/dim_content.parquet'"

# Filters live INSIDE each month's aggregation (HAVING) so the join sees ~75k rows, not millions.
df = con.sql(f"""
WITH mar AS (
  SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) AS imp_mar, SUM(gsc_clicks) AS clk_mar,
    SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions),0) AS pos_mar,
    COUNT(*) FILTER (gsc_impressions > 0) AS days_w_imp
  FROM {MAR} WHERE gsc_data_available IS TRUE GROUP BY 1,2
  HAVING SUM(gsc_impressions) >= 100
     AND SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions),0) BETWEEN 0.01 AND 20
),
apr AS (
  SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) AS imp_apr, SUM(gsc_clicks) AS clk_apr
  FROM {APR} WHERE gsc_data_available IS TRUE GROUP BY 1,2
  HAVING SUM(gsc_impressions) >= 50
)
SELECT m.*, a.imp_apr, a.clk_apr,
  d.content_created_date, d.content_updated_date, d.content_type, d.word_count
FROM mar m JOIN apr a USING (client_hash_id, content_hash_id)
JOIN {DIM} d USING (client_hash_id, content_hash_id)
""").df()
con.close()  # release DuckDB's memory before modeling

df["ctr_mar"] = 100 * df.clk_mar / df.imp_mar          # ctr as x100 percentage
df["ctr_apr"] = 100 * df.clk_apr / df.imp_apr
df["label"] = (df.ctr_apr < 0.8 * df.ctr_mar).astype(int)  # observed future outcome

df["content_age_days"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(df.content_created_date)).dt.days
df["days_since_update"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(df.content_updated_date)).dt.days.fillna(-1)
df["has_word_count"] = df.word_count.notna().astype(int)
df["word_count_f"] = df.word_count.fillna(0)

X_all = pd.concat([df[["imp_mar", "ctr_mar", "pos_mar", "days_w_imp", "content_age_days",
                       "days_since_update", "has_word_count", "word_count_f"]],
                   pd.get_dummies(df.content_type, prefix="type")], axis=1)
X_all["imp_mar"] = np.log1p(X_all["imp_mar"])
y = df["label"]

print(f"feature vector: {X_all.shape[0]:,} items x {X_all.shape[1]} features | "
      f"{df.client_hash_id.nunique()} clients")
print(f"label = next-30d CTR drop >=20% | base rate: {y.mean():.3f}")
print(list(X_all.columns))

feature vector: 73,527 items x 11 features | 41 clients
label = next-30d CTR drop >=20% | base rate: 0.419
['imp_mar', 'ctr_mar', 'pos_mar', 'days_w_imp', 'content_age_days', 'days_since_update', 'has_word_count', 'word_count_f', 'type_comparison article', 'type_feedly article', 'type_keyword article']


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing | Available when? |
|---|---|---|---|
| `imp_mar` (log1p) | March search visibility | never missing (aggregated) | Apr 1 — sums daily facts dated <= Mar 31 |
| `ctr_mar` | March click capture (x100 %) | never (imp >= 100 guaranteed) | Apr 1 — same window |
| `pos_mar` | impression-weighted avg position | never (filtered to 0-20) | Apr 1 — same window |
| `days_w_imp` | days of March with >=1 impression | never | Apr 1 — counts past days |
| `content_age_days` | days since creation at Mar 31 | never in this slice | static metadata, fixed at publication |
| `days_since_update` | days since last content update | filled with -1 sentinel | static metadata as of the snapshot |
| `has_word_count` | word count was measured at all | is itself the missingness flag | static |
| `word_count_f` | word count, 0 where unmeasured | 0-fill, honest **because** the flag rides along | static |
| `type_*` (one-hot) | content type | never | static |

**Why the flag matters (proved below):** word_count is missing for **25.7% of keyword articles but 0% of comparison articles** — the gap follows content type. A blind `fillna(0)` would teach the model "0 words" as a disguised category signal; the flag carries that signal openly instead.

In [4]:
# Proof: missingness is patterned by category, not random.
print("share of items with word_count missing, by content type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean()).round(3).to_string())
print()
print(f"base rate to beat everywhere below: {y.mean():.3f} (never judge a score without it)")

share of items with word_count missing, by content type:
content_type
comparison article    0.000
feedly article        0.013
keyword article       0.257

base rate to beat everywhere below: 0.419 (never judge a score without it)


## 3. The leakage hunt

Attacking my own features, one taxonomy entry at a time (leakage skill):

1. **Honest reference, on an honest split.** Random splits let a model memorize clients, so the real number is the **client-grouped** one — no client appears in both train and test. Both are reported; the *gap between them* measures memorization.
2. **Attack 1 — label-derived feature:** add `ctr_apr` (the label is literally computed from it). If the harness is working, AUC should leap toward 1.0. Then it's removed.
3. **Attack 2 — overlapping window:** add a "61-day CTR" spanning March 1 - April 30. Looks like an innocent long-window feature; it *contains the label window*, so it smuggles the outcome in. Also removed.
4. **Importance sanity check:** `ctr_mar` dominates the honest model (~0.78). Investigated, not celebrated: the label is *relative to* March CTR (a high-CTR page has more room to drop 20%), so March CTR legitimately carries signal about decline risk — mean reversion, not leakage. It comes strictly from the feature window.

**Attack checklist:** timeline drawn (features <= Mar 31, label = April) ✔ · no label-derived columns in the kept set ✔ · no product flags (none shipped, none reproduced) ✔ · population selection discloses its April-impressions filter ✔ · client-grouped split ✔ · base rate printed ✔ · metrics out-of-sample only ✔ · both deliberate leaks added, confessed, and deleted ✔

In [6]:
def fit_auc(Xtr, Xte, ytr, yte):
    m = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    return roc_auc_score(yte, m.predict_proba(Xte)[:, 1]), m

# Honest model — random split vs client-grouped split
Xtr, Xte, ytr, yte = train_test_split(X_all, y, test_size=0.3, random_state=42, stratify=y)
auc_rand, _ = fit_auc(Xtr, Xte, ytr, yte)

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
              .split(X_all, y, groups=df.client_hash_id))
auc_grp, m_grp = fit_auc(X_all.iloc[tr], X_all.iloc[te], y.iloc[tr], y.iloc[te])

print(f"base rate:                          {y.mean():.3f}")
print(f"honest AUC, random split:           {auc_rand:.3f}")
print(f"honest AUC, client-grouped split:   {auc_grp:.3f}   <- the number that counts")
print(f"memorization gap:                   {auc_rand - auc_grp:+.3f}")
print()
print("top-5 importances (grouped model):")
for c, v in sorted(zip(X_all.columns, m_grp.feature_importances_), key=lambda t: -t[1])[:5]:
    print(f"  {c:20s} {v:.3f}")

# Attack 1 — label-derived: ctr_apr IS the label's ingredient
X1 = X_all.copy(); X1["ctr_apr"] = df.ctr_apr
auc1, _ = fit_auc(X1.iloc[tr], X1.iloc[te], y.iloc[tr], y.iloc[te])

# Attack 2 — overlapping window: 61-day CTR spans into the label window
X2 = X_all.copy(); X2["ctr_61d"] = 100 * (df.clk_mar + df.clk_apr) / (df.imp_mar + df.imp_apr)
auc2, _ = fit_auc(X2.iloc[tr], X2.iloc[te], y.iloc[tr], y.iloc[te])

del X1, X2  # both leaks deleted — the honest number stands
print()
print(f"ATTACK 1  + ctr_apr (label-derived):      AUC {auc_grp:.3f} -> {auc1:.3f}   confession")
print(f"ATTACK 2  + ctr_61d (window overlap):     AUC {auc_grp:.3f} -> {auc2:.3f}   subtler, still guilty")
print(f"kept: the honest client-grouped AUC = {auc_grp:.3f}")

base rate:                          0.419
honest AUC, random split:           0.834
honest AUC, client-grouped split:   0.814   <- the number that counts
memorization gap:                   +0.020

top-5 importances (grouped model):
  ctr_mar              0.771
  imp_mar              0.095
  content_age_days     0.040
  pos_mar              0.033
  days_since_update    0.023

ATTACK 1  + ctr_apr (label-derived):      AUC 0.814 -> 0.993   confession
ATTACK 2  + ctr_61d (window overlap):     AUC 0.814 -> 0.951   subtler, still guilty
kept: the honest client-grouped AUC = 0.814


## 4. What I excluded and why

- **`ctr_apr`, `clk_apr`, `imp_apr`, any April column** — the label window; attack 1 shows what happens otherwise. (April is used once, transparently, in the disclosed population filter.)
- **Any multi-month aggregate crossing April** (e.g. the 61-day CTR of attack 2) — window overlap is leakage in a trench coat.
- **`ga4_*`, `sessions_*`, `scroll_events`** — measured on only ~4% of rows (w03 contract, Query 3); zero-fill means "not measured", and using it injects a measurement-gap signal.
- **`ai_*` columns** — sparse, and AI-referral analysis is a mentor-gated lane, not mine.
- **`client_hash_id`, `content_hash_id`, `keyword_hash_id`, `url_hash_id`** — pseudonyms; grouping/joining/splitting only. An ID in the features is memorization by construction.
- **`last_optimized_date`, `optimization_eligible_date`** — traces of the product's/editor's own decisions; a feature that encodes "someone already decided to act on this page" is a circular result.
- **`search_volume`, `competition`, `cpc`** — Week-1 finding: search volume's correlation with real impressions is ~0.001 in the starter slice; keyword-vendor fields also carry the patterned missingness without adding signal. Revisit only with evidence.
- **`trend_direction` / `trend_pct`-style derived movement fields** (starter CSV) — the original label trap from notebook 02; nothing derived from outcome movement enters the features, ever.

The cell below is the enforcement, not just the promise.

In [8]:
# Enforcement: assert nothing excluded is in the kept feature matrix.
banned_exact = {"ctr_apr", "clk_apr", "imp_apr", "ctr_61d", "client_hash_id", "content_hash_id",
                "keyword_hash_id", "url_hash_id", "search_volume", "competition", "cpc",
                "last_optimized_date", "optimization_eligible_date", "trend_direction", "trend_pct"}
banned_prefix = ("ga4_", "sessions_", "scroll_", "ai_")

violations = [c for c in X_all.columns
              if c in banned_exact or c.startswith(banned_prefix)]
assert not violations, f"excluded fields leaked into features: {violations}"
print(f"clean: none of the {len(banned_exact)} banned fields or {len(banned_prefix)} banned prefixes")
print(f"appear among the {X_all.shape[1]} kept features:")
print(list(X_all.columns))

clean: none of the 15 banned fields or 4 banned prefixes
appear among the 11 kept features:
['imp_mar', 'ctr_mar', 'pos_mar', 'days_w_imp', 'content_age_days', 'days_since_update', 'has_word_count', 'word_count_f', 'type_comparison article', 'type_feedly article', 'type_keyword article']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.